# Sentiment analysis 

## 1. Textblob-FR

Documentation: https://textblob.readthedocs.io/en/dev/

### Imports

In [1]:
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer

### Création d'une fonction `get_sentiment`

In [2]:
tb = Blobber(pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())

def get_sentiment(input_text):
    blob = tb(input_text)
    polarity, subjectivity = blob.sentiment
    polarity_perc = f"{100*abs(polarity):.0f}"
    subjectivity_perc = f"{100*subjectivity:.0f}"
    if polarity > 0:
        polarity_str = f"{polarity_perc}% positive"
    elif polarity < 0:
        polarity_str = f"{polarity_perc}% negative"
    else:
        polarity_str = "neutral"
    if subjectivity > 0:
        subjectivity_str = f"{subjectivity_perc}% subjective"
    else:
        subjectivity_str = "perfectly objective"
    print(f"This text is {polarity_str} and {subjectivity_str}.")

### Analyser le sentiment d'une phrase

In [3]:
get_sentiment("Ce journal est vraiment super intéressant.")

This text is 65% positive and 75% subjective.


In [4]:
get_sentiment("Cette phrase est négative et je ne suis pas content !")

This text is 41% negative and 60% subjective.


## 2. Utilisation de transformers

Documentation: https://github.com/TheophileBlard/french-sentiment-analysis-with-bert

**!!** Si le code ne tourne pas sur votre machine, vous pouvez le tester directement sur Google Colab en utilisant [ce lien](https://colab.research.google.com/github/TheophileBlard/french-sentiment-analysis-with-bert/blob/master/colab/french_sentiment_analysis_with_bert.ipynb) **!!**

Le modèle peut également être testé en ligne sur [HuggingFace](https://huggingface.co/tblard/tf-allocine)

Note : `transformers` v5 a supprimé le support TensorFlow. On utilise donc ici `philschmid/pt-tblard-tf-allocine`, un portage PyTorch du même modèle, avec les classes `AutoModelForSequenceClassification` (au lieu de `TFAutoModelForSequenceClassification`).

### Installation des librairies et imports

In [5]:
!uv pip install sentencepiece
!uv pip install "transformers>=5.19.0"
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

Using Python 3.13.15 environment at: c:\Users\DERRICK\tac\.venv
Checked 1 package in 61ms
Using Python 3.13.15 environment at: c:\Users\DERRICK\tac\.venv
Checked 1 package in 236ms
c:\Users\DERRICK\tac\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Chargement du modèle

In [6]:
tokenizer = AutoTokenizer.from_pretrained("philschmid/pt-tblard-tf-allocine")
model = AutoModelForSequenceClassification.from_pretrained("philschmid/pt-tblard-tf-allocine")

sentiment_analyser = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 28155.33it/s]


### Analyser le sentiment d'une phrase

In [7]:
sentiment_analyser("Ce journal est vraiment super intéressant.")

[{'label': 'POSITIVE', 'score': 0.985571026802063}]

In [8]:
sentiment_analyser("Cette phrase est négative et je ne suis pas content !")

[{'label': 'NEGATIVE', 'score': 0.9271842837333679}]

## Choisir 10 phrases dans les articles de 1960

In [13]:
import os
import re

# Lister les fichiers de l'année choisie
year = 1960
txt_path = "../../data/txt/"
files = sorted([f for f in os.listdir(txt_path) if f.endswith('.txt') and f"_{year}-" in f])

# Découper les textes en phrases et garder celles qui parlent des grands sujets de l'année
keywords = ["Congo", "Katanga", "Lumumba", "Tshombe", "Kennedy", "Khrouchtchev", "Van Looy"]
candidates = []
for f in files:
    with open(os.path.join(txt_path, f), 'r', encoding='utf-8') as fh:
        text = fh.read().replace("\n", " ")
    for sentence in re.split(r'(?<=[.!?])\s+', text):
        if 70 < len(sentence) < 220 and any(k in sentence for k in keywords):
            candidates.append((f, sentence))
len(candidates)

244

In [14]:
# Afficher les 50 premières phrases candidates, avec le fichier d'origine
for f, sentence in candidates[:50]:
    print(f, "|", sentence)

KB_JB838_1960-01-12_01-00011.txt | L'Anversols, nous l'avons déjà précisé, s’embarquera pour le Congo belge .à la .ftn du mois, en vue de s'aligner, en compagnie de Réné Baeten et de Nie Jan- seri, du côté belge, S ten Luridin, Bill N11®1 Son,: Smith;.
KB_JB838_1960-01-12_01-00011.txt | L’homme qui doute do, Van;iLooy : .— ^ L, *5 - : : Il est 'difficile; de parler .cyclisme sans prononcer le nom de TtlkuVan Looy.
KB_JB838_1960-01-12_01-00011.txt | Je crois même que j te n'aurais pas engage Van Looy a aller au’Tour de France, si je l’avais eu à mon service, et ceci .pour des raisons autres que celles, qui lé lui font, dédaigner maintenant.
KB_JB838_1960-01-13_01-00009.txt | MERCREDI 13 JANVIER 1960 - - M A ^ -2éH A F“ CYCLISME RI K VAN LOOY disputera toutes les classiques du printemps Rik Van Looy est actuellement en Italie.
KB_JB838_1960-01-13_01-00009.txt | Une chose est certaine, cependant' : Rik Van Looy a renoncé uu Tour d’Espagne et au Grand Prix Cyclomotoriste.
KB_JB838_1960-01-

In [15]:
# Les 10 phrases retenues (date de parution en commentaire)
phrases = [
    "Politiquement, la situation au Katanga paraît être la plus préoccupante.",  # 10/07/1960
    "Tshombe a fait, en outre, connaître son intention de proclamer l'indépendance du Katanga.",  # 10/07/1960
    "Les consuls britannique et américain ont donné l’ordre à leurs concitoyens de quitter le Congo aussitôt que possible.",  # 10/07/1960
    "Je crois que, sur le plan actuel, le problème du Congo est devenu un problème international.",  # 05/08/1960
    "Ils savent que Lumumba a été brillamment reçu aux Etats-Unis et que le monde entier l’écoute.",  # 05/08/1960
    "Il est heureux que le ministre Gilson et Cometro aient envisagé depuis longtemps les urgences d’intervenir militairement au Congo.",  # 05/08/1960
    "Dans l’immédiat, il est impossible de recruter pour le Congo des médecins à l’étranger, les conditions de travail au Congo étant inconnues, et la situation pouvant encore s'aggraver dans quelques semaines.",  # 05/08/1960
    "Le domaine dans lequel M Khrouchtchev et ses collègues semblent avoir les idées les plus nettes pour l’avenir est celui des relations du communisme avec le monde extérieur.",  # 16/03/1960
    "Au cours d’une conférence de presse, le sénateur Kennedy, très détendu, a déclaré qu'il était sûr de remporter la victoire, au plus tard au deuxième tour, compte tenu des voix qui lui avaient été promises.",  # 13/07/1960
    "Venant tout de suite après sa vibrante démonstration de la veille à Forli, cette victoire de Van Looy a confirmé l’excellente condition qu’il détient actuellement.",  # 27/05/1960
]
len(phrases)

10

In [ ]:
# Fichier d'origine de chaque phrase retenue
for phrase in phrases:
    sources = [f for f, sentence in candidates if sentence == phrase]
    print(sources, "|", phrase[:60])

['KB_JB838_1960-07-10_01-00001.txt'] | Politiquement, la situation au Katanga paraît être la plus p
['KB_JB838_1960-07-10_01-00001.txt'] | Tshombe a fait, en outre, connaître son intention de proclam
['KB_JB838_1960-07-10_01-00001.txt'] | Les consuls britannique et américain ont donné l’ordre à leu
['KB_JB838_1960-08-05_01-00007.txt'] | Je crois que, sur le plan actuel, le problème du Congo est d
['KB_JB838_1960-08-05_01-00007.txt'] | Ils savent que Lumumba a été brillamment reçu aux Etats-Unis
['KB_JB838_1960-08-05_01-00007.txt'] | Il est heureux que le ministre Gilson et Cometro aient envis
['KB_JB838_1960-08-05_01-00007.txt'] | Dans l’immédiat, il est impossible de recruter pour le Congo
['KB_JB838_1960-03-16_01-00007.txt'] | Le domaine dans lequel M Khrouchtchev et ses collègues sembl
['KB_JB838_1960-07-13_01-00006.txt'] | Au cours d’une conférence de presse, le sénateur Kennedy, tr
['KB_JB838_1960-05-27_01-00007.txt'] | Venant tout de suite après sa vibrante démonstration de la v


## Polarité et subjectivité des 10 phrases

In [17]:
# Analyse de chaque phrase avec la fonction du cours
for phrase in phrases:
    print(phrase)
    get_sentiment(phrase)
    print()

Politiquement, la situation au Katanga paraît être la plus préoccupante.
This text is 15% positive and 10% subjective.

Tshombe a fait, en outre, connaître son intention de proclamer l'indépendance du Katanga.
This text is neutral and perfectly objective.

Les consuls britannique et américain ont donné l’ordre à leurs concitoyens de quitter le Congo aussitôt que possible.
This text is 6% positive and 10% subjective.

Je crois que, sur le plan actuel, le problème du Congo est devenu un problème international.
This text is 22% positive and 5% subjective.

Ils savent que Lumumba a été brillamment reçu aux Etats-Unis et que le monde entier l’écoute.
This text is 15% positive and 20% subjective.

Il est heureux que le ministre Gilson et Cometro aient envisagé depuis longtemps les urgences d’intervenir militairement au Congo.
This text is 70% positive and 80% subjective.

Dans l’immédiat, il est impossible de recruter pour le Congo des médecins à l’étranger, les conditions de travail au Cong

In [18]:
import pandas as pd

# Résumer les résultats dans un tableau
pd.set_option('display.max_colwidth', None)
results = []
for phrase in phrases:
    polarity, subjectivity = tb(phrase).sentiment
    results.append({"Phrase": phrase, "Polarité": round(polarity, 2), "Subjectivité": round(subjectivity, 2)})
table = pd.DataFrame(results)
table

,Phrase,Polarité,Subjectivité
0,"Politiquement, la situation au Katanga paraît être la plus préoccupante.",0.15,0.10
1,"Tshombe a fait, en outre, connaître son intention de proclamer l'indépendance du Katanga.",0.00,0.00
2,Les consuls britannique et américain ont donné l’ordre à leurs concitoyens de quitter le Congo aussitôt que possible.,0.06,0.10
3,"Je crois que, sur le plan actuel, le problème du Congo est devenu un problème international.",0.22,0.05
4,Ils savent que Lumumba a été brillamment reçu aux Etats-Unis et que le monde entier l’écoute.,0.15,0.20
5,Il est heureux que le ministre Gilson et Cometro aient envisagé depuis longtemps les urgences d’intervenir militairement au Congo.,0.70,0.80
6,"Dans l’immédiat, il est impossible de recruter pour le Congo des médecins à l’étranger, les conditions de travail au Congo étant inconnues, et la situation pouvant encore s'aggraver dans quelques semaines.",-0.09,0.27
7,Le domaine dans lequel M Khrouchtchev et ses collègues semblent avoir les idées les plus nettes pour l’avenir est celui des relations du communisme avec le monde extérieur.,0.20,0.13
8,"Au cours d’une conférence de presse, le sénateur Kennedy, très détendu, a déclaré qu'il était sûr de remporter la victoire, au plus tard au deuxième tour, compte tenu des voix qui lui avaient été promises.",0.08,0.25
9,"Venant tout de suite après sa vibrante démonstration de la veille à Forli, cette victoire de Van Looy a confirmé l’excellente condition qu’il détient actuellement.",0.36,0.50
